In [4]:

import random
import math
from itertools import combinations

# ------------------------------------------
# 1. GET LOCATION DETAILS
# ------------------------------------------

num_regions = int(input("Enter number of farm regions: "))

locations = {}

locations["S"] = tuple(map(float, input(
    "Enter charging station coordinates (x y): "
).split()))

regions = []

for i in range(num_regions):
    name = chr(65 + i)

    coordinates = tuple(map(float, input(
        f"Enter coordinates of region {name} (x y): "
    ).split()))

    locations[name] = coordinates
    regions.append(name)

all_locations = ["S"] + regions

# ------------------------------------------
# 2. GET BATTERY DETAILS
# ------------------------------------------

battery_capacity = float(input("Enter battery capacity: "))
energy_per_km = float(input("Enter energy consumption per km: "))

# ------------------------------------------
# 3. CALCULATE DISTANCES
# ------------------------------------------

distance = {}

for i in all_locations:
    for j in all_locations:

        x1, y1 = locations[i]
        x2, y2 = locations[j]

        distance[i, j] = math.sqrt(
            (x2 - x1)**2 + (y2 - y1)**2
        )

# ------------------------------------------
# 4. ENTER WEATHER CONDITIONS
# ------------------------------------------

print("\nWEATHER CONDITIONS")
print("1 - Normal")
print("2 - Light Rain")
print("3 - Heavy Rain")
print("4 - Strong Wind")
print("5 - Storm / Hurricane")

# Unsafe weather conditions block a connection.
# Light rain is allowed but consumes more energy.

weather_multiplier = {
    1: 1.0,
    2: 1.1,
    3: None,
    4: None,
    5: None
}

weather = {}

for i, j in combinations(all_locations, 2):

    while True:
        try:
            condition = int(input(
                f"Weather between {i} and {j} (1-5): "
            ))

            if condition in weather_multiplier:
                break

            print("Enter a number from 1 to 5.")

        except ValueError:
            print("Please enter a valid number.")

    weather[i, j] = condition
    weather[j, i] = condition

# ------------------------------------------
# 5. CHECK WHETHER A CONNECTION IS SAFE
# ------------------------------------------

def is_safe(i, j):

    condition = weather[i, j]

    return weather_multiplier[condition] is not None


def get_energy(i, j):

    condition = weather[i, j]

    multiplier = weather_multiplier[condition]

    if multiplier is None:
        return float("inf")

    return (
        distance[i, j]
        * energy_per_km
        * multiplier
    )

# ------------------------------------------
# 6. INITIALIZE ACO PARAMETERS
# ------------------------------------------

num_ants = int(input("\nEnter number of ants: "))
max_iterations = int(input("Enter maximum iterations: "))

alpha = 1.0
beta = 2.0
evaporation = 0.5
Q = 100

pheromone = {}

for i in all_locations:
    for j in all_locations:
        pheromone[i, j] = 1.0

best_route = None
best_distance = float("inf")
best_energy = float("inf")

# ------------------------------------------
# 7. ACO MAIN LOOP
# ------------------------------------------

for iteration in range(max_iterations):

    valid_routes = []

    for ant in range(num_ants):

        current = "S"
        unvisited = regions.copy()
        route = ["S"]

        total_distance = 0
        total_energy = 0
        feasible = True

        # Construct a route
        while unvisited:

            allowed = []

            for next_region in unvisited:

                # Avoid unsafe weather connections
                if not is_safe(current, next_region):
                    continue

                if not is_safe(next_region, "S"):
                    continue

                leg_energy = get_energy(current, next_region)

                return_energy = get_energy(next_region, "S")

                required_energy = (
                    total_energy
                    + leg_energy
                    + return_energy
                )

                # Check battery availability
                if required_energy <= battery_capacity:
                    allowed.append(next_region)

            # No safe next region available
            if not allowed:
                feasible = False
                break

            # Calculate transition probabilities
            probabilities = []

            for region in allowed:

                tau = pheromone[current, region]

                # Distance and weather-adjusted energy
                cost = get_energy(current, region)

                eta = 1 / (cost + 0.001)

                probabilities.append(
                    (tau ** alpha) * (eta ** beta)
                )

            total_probability = sum(probabilities)

            probabilities = [
                p / total_probability for p in probabilities
            ]

            # Select next region
            next_region = random.choices(
                allowed,
                weights=probabilities,
                k=1
            )[0]

            # Update route details
            leg_distance = distance[current, next_region]
            leg_energy = get_energy(current, next_region)

            total_distance += leg_distance
            total_energy += leg_energy

            route.append(next_region)
            unvisited.remove(next_region)
            current = next_region

        # Return to charging station
        if feasible and is_safe(current, "S"):

            return_distance = distance[current, "S"]
            return_energy = get_energy(current, "S")

            total_distance += return_distance
            total_energy += return_energy

            if total_energy <= battery_capacity:

                route.append("S")

                valid_routes.append(
                    (route, total_distance, total_energy)
                )

                # Update best route
                if total_distance < best_distance:

                    best_distance = total_distance
                    best_energy = total_energy
                    best_route = route.copy()

    # --------------------------------------
    # 8. PHEROMONE EVAPORATION
    # --------------------------------------

    for edge in pheromone:
        pheromone[edge] *= (1 - evaporation)

    # --------------------------------------
    # 9. PHEROMONE DEPOSIT
    # --------------------------------------

    for route, route_distance, route_energy in valid_routes:

        deposit = Q / route_distance

        for i in range(len(route) - 1):

            edge = (route[i], route[i + 1])
            reverse_edge = (route[i + 1], route[i])

            pheromone[edge] += deposit
            pheromone[reverse_edge] += deposit

# ------------------------------------------
# 10. DISPLAY FINAL RESULT
# ------------------------------------------

print("\n========== DRONE PATH OPTIMIZATION ==========")

if best_route is not None:

    print("Best Route:", " -> ".join(best_route))

    print("Total Flight Distance:",
          round(best_distance, 2))

    print("Energy Consumption:",
          round(best_energy, 2))

    print("Battery Capacity:", battery_capacity)

    print("Remaining Battery:",
          round(battery_capacity - best_energy, 2))

    print("Route Status: SAFE AND FEASIBLE")

else:

    print("No feasible route found.")

    print("Possible reasons:")
    print("- Unsafe weather conditions")
    print("- Insufficient battery capacity")
    print("- No safe path to visit all regions and return")

    print("Do not start the mission with this route.")


Enter number of farm regions: 2
Enter charging station coordinates (x y): 4 4
Enter coordinates of region A (x y): 1 1
Enter coordinates of region B (x y): 1 4
Enter battery capacity: 120
Enter energy consumption per km: 10

WEATHER CONDITIONS
1 - Normal
2 - Light Rain
3 - Heavy Rain
4 - Strong Wind
5 - Storm / Hurricane
Weather between S and A (1-5): 1
Weather between S and B (1-5): 1
Weather between A and B (1-5): 1

Enter number of ants: 5
Enter maximum iterations: 4

========== DRONE PATH OPTIMIZATION ==========
Best Route: S -> B -> A -> S
Total Flight Distance: 10.24
Energy Consumption: 102.43
Battery Capacity: 120.0
Remaining Battery: 17.57
Route Status: SAFE AND FEASIBLE
